In [6]:
import importlib
import sys
import os
import pickle

# performance imports for torch: torch kernel uses one core only.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch
import pandas as pd

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')

In [7]:
# Event log and dataset configuration (must match the base loader)
event_log_location = '../../../simulator/procurement_event_log.csv'
result_name = 'procurement_all'
min_suffix_size = 5

event_log_properties = {
    'case_name': 'case:concept:name',
    'concept_name': 'concept:name',
    'timestamp_name': 'time:timestamp',
}

## Load artifacts from base loader and decision mining

In [8]:

# Load the Petri net discovered in the base loader
petri_net_path = '../../../../data/Procurement/Petri_net/procurement.pkl'
with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)
print('Petri net loaded from', petri_net_path)

# Load the normal (non-decision-labeled) tensor datasets from the base loader
train_set = torch.load('../../../../data/Procurement/tensor_data/normal/procurement_all_5_train.pkl', weights_only=False)
val_set   = torch.load('../../../../data/Procurement/tensor_data/normal/procurement_all_5_val.pkl', weights_only=False)
print(f'Train set: {len(train_set)} prefixes, Val set: {len(val_set)} prefixes')

# Load case IDs from saved CSVs
train_pref_df = pd.read_csv('../../../../data/Procurement/raw_data/procurement_all_5_train.csv')
val_pref_df   = pd.read_csv('../../../../data/Procurement/raw_data/procurement_all_5_val.csv')

unique_list_train = train_pref_df['case:concept:name'].dropna().unique().tolist()
unique_list_val   = val_pref_df['case:concept:name'].dropna().unique().tolist()
print(f'Unique train cases: {len(unique_list_train)}, val cases: {len(unique_list_val)}')


Petri net loaded from ../../../../data/Procurement/Petri_net/procurement.pkl
Train set: 83492 prefixes, Val set: 19284 prefixes
Unique train cases: 6500, val cases: 1500


## Decision-label the tensor-datasets

In [9]:

import data_processing.decision_labeling
importlib.reload(data_processing.decision_labeling)
from data_processing.decision_labeling import DecisionLabeler

from pm4py.objects.log.util import dataframe_utils
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignment_algo
from pm4py.utils import get_properties

# Decision mining artifact paths
decision_bundle_path = '../../../../data/Procurement/Petri_net/data_aware_Petri_net/decision_places_bundle.json'
decision_model_dir   = '../../../../data/Procurement/Petri_net/data_aware_Petri_net/models'

# Attributes used during decision mining (must match exactly)
# dynamic: actually change value within a trace
dynamic_attributes = ['org:resource',
                      'amount',
                      'budget_status',
                      'revision_count',
                      'supplier_type',
                      'goods_match',
                      'invoice_deviation_pct',
                      'case_elapsed_time',
                      'event_elapsed_time']

# static: fixed for the entire trace, only vary across cases
static_attributes = ['requester_seniority',
                     'department',
                     'category',
                     'priority']

print("Decision-labeling dynamic attributes:", dynamic_attributes)
print("Decision-labeling static attributes:", static_attributes)


Decision-labeling dynamic attributes: ['org:resource', 'amount', 'budget_status', 'revision_count', 'supplier_type', 'goods_match', 'invoice_deviation_pct', 'case_elapsed_time', 'event_elapsed_time']
Decision-labeling static attributes: ['requester_seniority', 'department', 'category', 'priority']


In [10]:
# Load the original event log with timestamps and convert to pm4py column names
el_df = pd.read_csv(event_log_location)
el_df = el_df.rename(columns={
    event_log_properties['case_name']:      'case:concept:name',
    event_log_properties['concept_name']:   'concept:name',
    event_log_properties['timestamp_name']: 'time:timestamp',
})
el_df['time:timestamp'] = pd.to_datetime(el_df['time:timestamp'])
el_df = dataframe_utils.convert_timestamp_columns_in_df(el_df)
print(f"Event log: {len(el_df)} events, {el_df['case:concept:name'].nunique()} cases")

Event log: 128360 events, 10000 cases


In [11]:
# Compute optimal alignments for all case IDs across train / val
all_case_ids = list(dict.fromkeys(unique_list_train + unique_list_val))

# Filter event log to relevant cases
el_aligned = el_df[el_df['case:concept:name'].isin(all_case_ids)].copy()

params = get_properties(el_aligned)
params["ret_tuple_as_trans_desc"] = True

aligned_results = alignment_algo.apply(el_aligned, net, im, fm, parameters=params)
all_alignments = [r['alignment'] for r in aligned_results]

# Case-ID ordering must match the alignment list (pm4py preserves DataFrame group order)
sorted_case_ids = (el_aligned.drop_duplicates(subset='case:concept:name', keep='first')['case:concept:name'].tolist())
print(f"Computed {len(all_alignments)} alignments for {len(sorted_case_ids)} cases")

aligning log, completed variants ::   0%|          | 0/15 [00:00<?, ?it/s]

Computed 8000 alignments for 8000 cases


In [12]:
# Create decision labeler (replaces el_loader.label_dataset)
labeler = DecisionLabeler(petri_net=(net, im, fm),
                          decision_model_dir=decision_model_dir,
                          decision_places_bundle_path=decision_bundle_path,
                          dynamic_attributes=dynamic_attributes,
                          static_attributes=static_attributes)

# Decision-label train set (offline, from optimal alignments)
labeler.label_dataset_offline(train_set, el_aligned, sorted_case_ids, all_alignments)
print(f"Train set labeled: {len(train_set)} prefixes")

Train set labeled: 83492 prefixes


In [13]:
# Decision-label val set (offline)
labeler.label_dataset_offline(val_set, el_aligned, sorted_case_ids, all_alignments)
print(f"Val set labeled: {len(val_set)} prefixes")

Val set labeled: 19284 prefixes


In [14]:
# Identify the concept:name feature index for guard tensors
activity_feature_name = 'concept:name'
concept_name_feature_idx = next(i for i, cat in enumerate(train_set.all_categories[0]) if cat[0] == activity_feature_name)

# Convert sparse decision_data → dense guard tensors so they are stored in the .pkl
for name, ds in [("Train", train_set), ("Val", val_set)]:
    ds.prepare_guard_tensors(concept_name_feature_idx)
    
    # guard_targets is a tensor of shape [N, T, C] on the dataset, or [B, S, C] after batching/slicing. It stores a soft probability distribution z_i over next-event label classes for each event position.
    # guard_mask is a tensor of shape [N, T] or [B, S]. It is just an indicator: 1 where that event position has a usable decision label, 0 where it does not.
    # guard_confidence is a tensor of shape [N, T] on the dataset. It stores a scalar confidence c_i for each labeled event position, usually the maximum probability in the decision distribution if no explicit confidence was stored.
    
    print(f"{name}: guard_targets {ds._guard_targets.shape}, "
          f"guard_mask {ds._guard_mask.shape}, "
          f"guard_confidence {ds._guard_confidence.shape}")

Train: guard_targets torch.Size([83492, 29, 18]), guard_mask torch.Size([83492, 29]), guard_confidence torch.Size([83492, 29])
Val: guard_targets torch.Size([19284, 29, 18]), guard_mask torch.Size([19284, 29]), guard_confidence torch.Size([19284, 29])


In [15]:

# Save decision-labeled datasets
dl_dir = '../../../../data/Procurement/tensor_data/decision_labeled/'
os.makedirs(dl_dir, exist_ok=True)

sfx = '_' + str(train_set.min_suffix_size)
torch.save(train_set, dl_dir + result_name + sfx + '_train.pkl')
torch.save(val_set,   dl_dir + result_name + sfx + '_val.pkl')
print("Saved decision-labeled datasets to", dl_dir)


Saved decision-labeled datasets to ../../../../data/Procurement/tensor_data/decision_labeled/


In [16]:
# Verify: inspect a sample of decision labels
for name, ds in [("Train", train_set), ("Val", val_set)]:
    dd = ds.decision_data
    n = len(dd) if not isinstance(dd, torch.Tensor) else dd.size(0)
    print(f"\n{name}: {n} prefixes")
    if isinstance(dd, list) and n > 0:
        sample = dd[0]
        print(f"  First prefix decisions ({len(sample)} events):")
        for j, (place, dist, c_i) in enumerate(sample[:5]):
            print(f"e_{j}: place={place}, z_i={dist}, c_i={c_i}")


Train: 83492 prefixes
  First prefix decisions (6 events):
e_0: place=p_3, z_i={'Approve Requisition': 0.96794070084173, 'Reject Requisition': 0.03205929915827003}, c_i=0.96794070084173
e_1: place=⊥, z_i={}, c_i=0.0
e_2: place=⊥, z_i={}, c_i=0.0
e_3: place=⊥, z_i={}, c_i=0.0
e_4: place=⊥, z_i={}, c_i=0.0

Val: 19284 prefixes
  First prefix decisions (6 events):
e_0: place=p_3, z_i={'Approve Requisition': 0.9841403077906753, 'Reject Requisition': 0.015859692209324684}, c_i=0.9841403077906753
e_1: place=⊥, z_i={}, c_i=0.0
e_2: place=⊥, z_i={}, c_i=0.0
e_3: place=⊥, z_i={}, c_i=0.0
e_4: place=⊥, z_i={}, c_i=0.0


In [17]:
# Example output of one train sample
print(train_set.all_categories[0][0][2])
print(train_set.categorical_tensors[0][0])
print(train_set.decision_data[0])

{'Approve Invoice': 1, 'Approve Requisition': 2, 'Close Case': 3, 'Collect Quotations': 4, 'Create Purchase Order': 5, 'Create Purchase Requisition': 6, 'EOS': 7, 'Evaluate Quotations': 8, 'Flag Invoice Mismatch': 9, 'Pay Invoice': 10, 'Receive Goods': 11, 'Reject Requisition': 12, 'Reorder Goods': 13, 'Request Credit Note': 14, 'Revise Requisition': 15, 'Select Supplier': 16, 'Send Purchase Order': 17}
tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  6,  2,  4,  8, 16,  5])
[('p_3', {'Approve Requisition': 0.96794070084173, 'Reject Requisition': 0.03205929915827003}, 0.96794070084173), ('⊥', {}, 0.0), ('⊥', {}, 0.0), ('⊥', {}, 0.0), ('⊥', {}, 0.0), ('⊥', {}, 0.0)]
